# HaroCLIP — reframe + captioning on Colab, reusing an existing highlight selection

Re-runs the **reframe** (dynamic multi-speaker-follow, added 2026-08-01) and
**captioning** stages on Colab for a job whose highlight selection already
exists in `data/haroclip.db` (`HighlightClip.segments_json`) — **without
calling the Claude API again**.

The DB alone isn't enough to resume: `run_reframe()`/`run_captioning()`
check DB status only, not whether their input *files* exist on disk. Since
`data/clips/<job_id>/clip_XX.mp4` (reframe's input) and
`data/videos/<job_id>/transcript.json` (captioning's input) aren't available
locally for this job, this notebook regenerates them first — re-downloading
the source video from `IngestionJob.source_url` and re-transcribing with
whisper, then re-rendering each clip from the **already-selected**
`segments_json` in the DB. No highlight *selection* step runs, so the Claude
API is never called.

**Before running**: put `src/` and `data/haroclip.db` (a copy of
`data/VAST/haroclip.db`) under one folder in your Google Drive, e.g.
`MyDrive/HaroCLIP/{src/, data/haroclip.db}`, and set `DRIVE_ROOT`/`JOB_ID`
in the config cell below.

**Run cells top to bottom, in order, once per fresh runtime.** The env-var
cell must run before any cell that imports from `src` — `src/utils/db.py`
reads `DATA_DIR` once at import time, so importing out of order binds it to
the wrong path for the rest of the session (fix: Runtime → Restart runtime,
then run in order).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config + env vars — must run before any `src` import

Edit `DRIVE_ROOT`/`JOB_ID` to match your Drive layout and target job.

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/HaroCLIP"  # folder containing src/ and data/haroclip.db
JOB_ID = "fa5be0db-ff05-4f3f-b3eb-eccc9af8915d"  # ingestion_job_id to resume

DATA_DIR = f"{DRIVE_ROOT}/data"
os.environ["DATA_DIR"] = DATA_DIR
os.environ["HF_HUB_DISABLE_XET"] = "1"
# YOLOV8_FACE_WEIGHTS_PATH has its own hardcoded default ("data/models/...")
# independent of DATA_DIR (src/detection/face_detector.py) -- must be set
# explicitly so it resolves under the Drive-mounted data/ tree too.
os.environ["YOLOV8_FACE_WEIGHTS_PATH"] = f"{DATA_DIR}/models/yolov8x-face-lindevs.pt"

print("DATA_DIR =", os.environ["DATA_DIR"])

## 3. System dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg sqlite3

## 4. Python dependencies

Mirrors `requirements.txt`'s active deps plus the heavy ML packages that stay
commented out there (installed separately, same convention as
`scripts/entrypoint.sh`). `torch`/`torchvision` are **not** reinstalled —
Colab's runtime already ships a CUDA-matched build.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pydantic python-dotenv httpx python-multipart \
    ffmpeg-python opencv-python numpy anthropic yt-dlp sqlalchemy
!pip install -q faster-whisper ultralytics supervision python_speech_features
# ctranslate2 (faster-whisper's backend) version compatibility on Colab has
# been a moving target across test sessions -- Colab's assigned GPU host
# (and its driver/CUDA/cuDNN bundle) can differ between sessions, so no
# single pin has proven reliable yet. Left unpinned here (whatever
# faster-whisper's own dependency resolution picks) since a driver seen in
# testing (580.82.07, CUDA 13.0 -- see `!nvidia-smi`) comfortably supports
# recent ctranslate2 releases. If step 9 (re-transcribe) fails loading the
# whisper model, see the troubleshooting cell right below this one before
# reaching for a pin -- diagnose first, since the *direction* to pin
# (older vs newer) depends on which specific host you landed on this
# session.

If step 9 (re-transcribe) fails on whisper/`ctranslate2` load, the exact
error message tells you which direction to pin -- Colab's assigned GPU host
varies session to session, so there's no one fix that always works. Run
these diagnostics first:
```python
# !nvidia-smi   # look at the "CUDA Version: X.Y" in the header -- the max
#               # CUDA runtime the driver actually supports right now
# !pip show ctranslate2 | grep Version
```
Then, back in section 4's dependency cell, add ONE of these right after the
`faster-whisper` install line depending on the error you actually saw, and
re-run from there (restart runtime first -- `ctranslate2` is a native
extension, pip-reinstalling it in an already-running kernel has no effect
until the process restarts):
```python
# "CUDA driver version is insufficient for CUDA runtime version"
# -> ctranslate2 needs a newer CUDA runtime than this driver supports, pin OLDER:
# !pip install -q "ctranslate2==4.4.0"

# "Could not load library libcudnn_ops_infer.so.8" (or similar .so.8 file)
# -> ctranslate2 wants cuDNN8 but this environment only has cuDNN9, pin NEWER:
# !pip install -q "ctranslate2==4.5.0"
```
As a last resort (works, but no GPU/faster inference for transcription --
everything else in the pipeline still uses the GPU normally), force
CPU-only whisper for just this run:
```python
# import os
# os.environ["WHISPER_DEVICE"] = "cpu"
# os.environ["WHISPER_COMPUTE_TYPE"] = "int8"
```
run this before step 9's cell (must be set before `transcribe()` is
called, same ordering caveat as `DATA_DIR`).

Separately, if a cell fails on `libcublas.so.12: cannot open shared object
file`, run this (same fix as `entrypoint.sh` step 4 — usually not needed on
Colab, Colab's own torch build typically already resolves this):
```python
# import subprocess
# cublas_cudnn_path = subprocess.check_output([
#     "python3", "-c",
#     "import os, nvidia.cublas.lib, nvidia.cudnn.lib; "
#     "print(os.path.dirname(nvidia.cublas.lib.__file__) + ':' + os.path.dirname(nvidia.cudnn.lib.__file__))",
# ]).decode().strip()
# os.environ["LD_LIBRARY_PATH"] = cublas_cudnn_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
```

## 5. Put `src/` on the import path

In [ ]:
import sys

if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)  # parent of src/, so `from src.xxx import yyy` resolves

import src  # noqa: F401 -- sanity check the import path is correct
print("src imported from:", src.__file__)

## 6. YOLOv8-face weights (downloaded once, persisted on Drive)

In [ ]:
import pathlib

weights_path = pathlib.Path(os.environ["YOLOV8_FACE_WEIGHTS_PATH"])
weights_path.parent.mkdir(parents=True, exist_ok=True)
if not weights_path.exists():
    print("downloading YOLOv8-face xlarge weights...")
    !curl -L "https://github.com/lindevs/yolov8-face/releases/latest/download/yolov8x-face-lindevs.pt" -o "{weights_path}"
else:
    print("weights already present:", weights_path)

## 7. Sanity check: open the DB, confirm the job's status

In [ ]:
from src.highlights.models import HighlightClip, HighlightJob
from src.ingestion.models import IngestionJob
from src.processing.models import ProcessingJob
from src.utils.db import DATA_DIR, SessionLocal, init_db

print("resolved DATA_DIR:", DATA_DIR)
assert str(DATA_DIR) == os.environ["DATA_DIR"], (
    "DATA_DIR was already bound before this cell ran -- restart the runtime "
    "and run cells in order starting from the config cell."
)

init_db()
db = SessionLocal()

ingestion_job = db.get(IngestionJob, JOB_ID)
assert ingestion_job is not None, f"no ingestion job {JOB_ID} in this DB"
print("ingestion job:", ingestion_job.title, "|", ingestion_job.status.value, "|", ingestion_job.source_url)

processing_job = db.query(ProcessingJob).filter_by(ingestion_job_id=JOB_ID).one()
print("processing job:", processing_job.status.value)

highlight_job = db.query(HighlightJob).filter_by(ingestion_job_id=JOB_ID).one()
highlight_clips = (
    db.query(HighlightClip).filter_by(highlight_job_id=highlight_job.id).order_by(HighlightClip.rank).all()
)
print("highlight job:", highlight_job.status.value, "| clips:", len(highlight_clips))
for c in highlight_clips:
    print(f"  #{c.rank:02d} [{c.start_seconds:.1f}-{c.end_seconds:.1f}] -> {c.output_path}")

## 8. Recovery step A — re-download the source video

Pure yt-dlp + ffmpeg audio extraction, reusing the existing, tested
`run_processing()` — no GPU, no LLM. Regenerates files at the exact paths
already recorded in `processing_jobs` (deterministic from `JOB_ID`).

If this fails because YouTube blocks Colab's shared IP range, the fallback
is to download the source video by hand and place it at
`{DATA_DIR}/{processing_job.video_path}` yourself, then skip re-running this
cell.

In [ ]:
from src.processing.service import run_processing

processing_job = run_processing(db, JOB_ID, force=True)
print(processing_job.status.value, "| video:", processing_job.video_path, "| audio:", processing_job.audio_path)
assert processing_job.status.value == "ready", f"processing failed: {processing_job.error_stage} {processing_job.error_message}"

## 9. Recovery step B — re-transcribe, write `transcript.json`

Calls `transcribe()` directly (not `run_highlight_detection()`, which would
also re-call Claude and wipe the existing `HighlightClip` rows). Writes to
the exact path already recorded in `HighlightJob.transcript_path` — no DB
row needs updating, only the file needs to physically exist again.

In [ ]:
import dataclasses
import json
import logging

from src.transcription.transcriber import transcribe

logger = logging.getLogger("colab")
logging.basicConfig(level=logging.INFO)

audio_path = DATA_DIR / processing_job.audio_path
segments = transcribe(audio_path, logger=logger)

transcript_path = DATA_DIR / highlight_job.transcript_path
transcript_path.parent.mkdir(parents=True, exist_ok=True)
transcript_path.write_text(
    json.dumps([dataclasses.asdict(s) for s in segments]), encoding="utf-8"
)
print("wrote", transcript_path, "|", len(segments), "segments")

## 10. Recovery step C — re-render each clip from the existing `segments_json`

Uses the highlight *selection* already stored in the DB — no Claude call,
no changes to the `highlight_clips` rows. Pure ffmpeg, via the same
`render_clip()` the highlights stage itself uses.

In [ ]:
from src.rendering.clipper import render_clip

video_path = DATA_DIR / processing_job.video_path

for clip in highlight_clips:
    segs = json.loads(clip.segments_json)
    out_path = DATA_DIR / clip.output_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    render_clip(video_path, [(s["start"], s["end"]) for s in segs], out_path)
    print(f"#{clip.rank:02d} rendered -> {out_path}")

## 11. Run reframe (new dynamic multi-speaker-follow code)

In [ ]:
from src.reframe.service import run_reframe

for clip in highlight_clips:
    job = run_reframe(db, clip.id, force=True)
    status = job.status.value
    print(f"#{clip.rank:02d} {status}", job.output_path if status == "ready" else f"[{job.error_stage}] {job.error_message}")

## 12. Run captioning

In [ ]:
from src.captioning.service import run_captioning

for clip in highlight_clips:
    job = run_captioning(db, clip.id, force=True)
    status = job.status.value
    print(f"#{clip.rank:02d} {status}", job.output_path if status == "ready" else f"[{job.error_stage}] {job.error_message}")

## 13. Verify

List the final outputs and preview one inline — check whether a
previously-frozen-between-two-people clip now visibly pans to follow
whoever's speaking (watch for `speaker window: ...` log lines during step 11
showing more than one window for a multi-speaker clip).

In [ ]:
import pathlib

captioned_dir = pathlib.Path(DATA_DIR) / "captioned" / JOB_ID
outputs = sorted(captioned_dir.glob("*.mp4"))
for f in outputs:
    print(f)

In [ ]:
from IPython.display import Video

# Adjust the index to preview a different clip.
Video(str(outputs[0]), embed=True, width=360) if outputs else print("no outputs yet")